# Stabilizers and Syndrome Formalism

In the previous notebook, we studied the 3-qubit repetition code using parity checks and a simple decoding rule. In this notebook, we reformulate those ideas using the *stabilizer formalism*, which provides a general framework for quantum error correction.

We will:
* define stabilizers and code spaces,
* reinterpret the repetition code in this language,
* understand how errors are detected via commutation relations,
* and connect syndromes to stabilizer eigenvalues.

This provides the conceptual foundation for more advanced codes such as the surface code.

## What is a stabilizer?

A stabilizer is an operator \(S\) that leaves a quantum state unchanged:

$$
S|\psi\rangle = |\psi\rangle.
$$

A stabilizer code is defined by a set of commuting operators $ \{S_i\} $. The code space consists of all states that are +1 eigenstates of every stabilizer:

$$
S_i |\psi\rangle = |\psi\rangle \quad \forall i.
$$

This defines a subspace of the full Hilbert space in which logical information is stored.

In [1]:
import numpy as np

# Define Pauli matrices
I = np.array([[1, 0], [0, 1]])
X = np.array([[0, 1], [1, 0]])
Z = np.array([[1, 0], [0, -1]])

def kron3(a, b, c):
    return np.kron(np.kron(a, b), c)

## Stabilizers of the repetition code

For the 3-qubit repetition code, the stabilizers are:

$$
S_1 = Z_1 Z_2, \qquad S_2 = Z_2 Z_3.
$$

These operators compare neighboring qubits in the computational basis.

The logical codewords $\ket{000}$ and $\ket{111}$ should be +1 eigenstates of both stabilizers.

In [2]:
# Build stabilizers
S1 = kron3(Z, Z, I)
S2 = kron3(I, Z, Z)

# Define basis states
zero = np.array([1, 0])
one = np.array([0, 1])

def kron_state(a, b, c):
    return np.kron(np.kron(a, b), c)

state_000 = kron_state(zero, zero, zero)
state_111 = kron_state(one, one, one)

# Verify

print("S1 |000> == |000>:", np.allclose(S1 @ state_000, state_000))
print("S2 |000> == |000>:", np.allclose(S2 @ state_000, state_000))

print("S1 |111> == |111>:", np.allclose(S1 @ state_111, state_111))
print("S2 |111> == |111>:", np.allclose(S2 @ state_111, state_111))

S1 |000> == |000>: True
S2 |000> == |000>: True
S1 |111> == |111>: True
S2 |111> == |111>: True


Both $\ket{000}$ and $\ket{111}$ are +1 eigenstates of the stabilizers. This means they lie in the code space defined by \(S_1\) and \(S_2\).

Thus, the repetition code can be defined entirely in terms of its stabilizers.

## Errors as operators

We now model errors as Pauli operators. A bit-flip error on qubit $i$ is represented by $X_i$.

We will examine how these errors affect stabilizer measurements.

In [3]:
# Define operators
X1 = kron3(X, I, I)
X2 = kron3(I, X, I)
X3 = kron3(I, I, X)

# Apply errors
error_state = X1 @ state_000
print("Error state (X1 applied to |000>):")
print(error_state)

# Measure stablizer eigenvalues
def stabilizer_sign(S, state):
    """Return +1 or -1 eigenvalue of stabilizer S on state."""
    val = S @ state
    return np.sign(np.vdot(state, val).real)

print("S1 eigenvalue after X1:", stabilizer_sign(S1, error_state))
print("S2 eigenvalue after X1:", stabilizer_sign(S2, error_state))

Error state (X1 applied to |000>):
[0 0 0 0 1 0 0 0]
S1 eigenvalue after X1: -1
S2 eigenvalue after X1: 1


## Commutation and anticommutation

The Pauli operators satisfy:

$$
XZ = -ZX.
$$

This minus sign is what causes stabilizer eigenvalues to flip.

For example:

- $X_1$ anticommutes with $Z_1$, so it flips $Z_1 Z_2$,
- but commutes with $Z_3$, so it leaves $Z_2 Z_3$ unchanged.

This algebraic property is the foundation of syndrome extraction.

In [4]:
# Check commutation: S1 X1 vs X1 S1
lhs = S1 @ X1
rhs = X1 @ S1

print("Do S1 and X1 commute?", np.allclose(lhs, rhs))
print("Do they anticommute?", np.allclose(lhs, -rhs))

Do S1 and X1 commute? False
Do they anticommute? True


## Syndrome as stabilizer eigenvalues

The syndrome bits are simply the signs of the stabilizer measurements:

- +1 → 0  
- −1 → 1  

Thus, the syndrome is the pattern of stabilizer eigenvalues after an error.

Each single-qubit bit-flip produces a unique pattern of sign flips, allowing us to identify the error.

## Dimension of the code space

For $n$ qubits, the Hilbert space has dimension $2^n$.

Each independent stabilizer constraint reduces the dimension by a factor of 2.

For the repetition code:
- $n = 3$,
- 2 stabilizers,

so the code space has dimension:

$$
2^{3-2} = 2.
$$

This corresponds to encoding a single logical qubit.